In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    xgb = None
    print("⚠️ XGBoost not installed. Install with: pip install xgboost")
    print("   Continuing without XGBoost models...")

import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All libraries imported successfully")


In [ ]:
BASE_DIR = Path(r"d:/sugarcare_backend")
if not BASE_DIR.exists():
    BASE_DIR = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()

DATASET_PATHS = {
    'main': BASE_DIR / "data" / "your_datasets" / "RiskForecasting.csv",
    'dataset2': BASE_DIR / "data" / "your_datasets" / "diabetes_012_health_indicators_BRFSS2015.csv",
    'dataset3': BASE_DIR / "data" / "your_datasets" / "diabetes_binary_5050split_health_indicators_BRFSS2015.csv",
    'dataset4': BASE_DIR / "data" / "your_datasets" / "diabetes_binary_health_indicators_BRFSS2015.csv",
    'diabetes_dataset': BASE_DIR / "data" / "your_datasets" / "diabetes_dataset.csv",
}

OUTPUT_DIR = Path(r"d:/sugarcare_backend/models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'target_column': 'risk_score',
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
}

FEATURE_COLUMNS = None 

RISK_THRESHOLDS = {
    'low': (0, 33),
    'moderate': (33, 66),
    'high': (66, 100)
}

MODELS_TO_TRAIN = ['random_forest', 'xgboost', 'gradient_boosting', 'logistic_regression']

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Datasets: {list(DATASET_PATHS.keys())}")
print(f"Target column: {CONFIG['target_column']}")
print(f"Test size: {CONFIG['test_size']}")
print(f"Output directory: {OUTPUT_DIR}")
print("=" * 60)


In [ ]:
def load_and_map_datasets(dataset_paths):
    """Load and map multiple CSV datasets to unified risk prediction format."""
    all_dfs = []
    
    for name, path in dataset_paths.items():
        if not path.exists():
            print(f"⚠️ Warning: Dataset '{name}' not found at {path}")
            continue
        
        print(f"Loading {name}...")
        try:
            df = pd.read_csv(path)
            print(f"  ✅ Loaded {len(df)} rows, {len(df.columns)} columns")
            print(f"  Columns: {list(df.columns)[:5]}...")
            
            mapped_df = map_dataset_to_unified_format(df, name)
            if mapped_df is not None:
                all_dfs.append(mapped_df)
                print(f"  ✅ Mapped {name} to unified format")
        except Exception as e:
            print(f"  ❌ Error loading {name}: {e}")
            continue
    
    if not all_dfs:
        raise ValueError("No datasets were loaded successfully!")
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n✅ Combined dataset: {len(combined_df)} rows, {len(combined_df.columns)} columns")
    
    return combined_df

def map_dataset_to_unified_format(df, dataset_name):
    """Map different dataset structures to unified risk prediction format."""
    mapped = pd.DataFrame()
    
    if dataset_name == 'diabetes_dataset':
        if 'Age' in df.columns:
            mapped['age'] = pd.to_numeric(df['Age'], errors='coerce')
        if 'Sex' in df.columns:
            mapped['sex'] = df['Sex'].map({'Male': 1, 'Female': 0, 'M': 1, 'F': 0}).fillna(0)
        if 'Fasting_Blood_Glucose' in df.columns:
            mapped['blood_glucose'] = pd.to_numeric(df['Fasting_Blood_Glucose'], errors='coerce')
        if 'HbA1c' in df.columns:
            mapped['hba1c'] = pd.to_numeric(df['HbA1c'], errors='coerce')
        if 'Cholesterol_Total' in df.columns:
            chol = pd.to_numeric(df['Cholesterol_Total'], errors='coerce')
            mapped['high_chol'] = (chol > 200).astype(float)
        if 'Physical_Activity_Level' in df.columns:
            activity_map = {'Low': 0.0, 'Moderate': 1.0, 'High': 2.0}
            mapped['phys_activity'] = df['Physical_Activity_Level'].map(activity_map).fillna(1.0)
        if 'Dietary_Intake_Calories' in df.columns:
            calories = pd.to_numeric(df['Dietary_Intake_Calories'], errors='coerce')
            mapped['diet'] = (calories > 2500).astype(float) * 0.9 + (calories < 1500).astype(float) * 0.3
            mapped['diet'] = mapped['diet'].fillna(0.6)
        
        risk_factors = []
        
        if 'hba1c' in mapped.columns:
            hba1c_risk = ((mapped['hba1c'] - 5.0) / 10.0 * 100).clip(0, 100)
            risk_factors.append(hba1c_risk)
        
        if 'blood_glucose' in mapped.columns:
            glucose_risk = ((mapped['blood_glucose'] - 100) / 100 * 50).clip(0, 100)
            risk_factors.append(glucose_risk)
        
        if 'high_chol' in mapped.columns:
            risk_factors.append(mapped['high_chol'] * 70)
        
        if 'Family_History_of_Diabetes' in df.columns:
            family_risk = pd.to_numeric(df['Family_History_of_Diabetes'], errors='coerce').fillna(0) * 30
            risk_factors.append(family_risk)
        
        if 'Previous_Gestational_Diabetes' in df.columns:
            gest_risk = pd.to_numeric(df['Previous_Gestational_Diabetes'], errors='coerce').fillna(0) * 25
            risk_factors.append(gest_risk)
        
        if 'phys_activity' in mapped.columns:
            activity_risk = (2.0 - mapped['phys_activity']) / 2.0 * 40
            risk_factors.append(activity_risk)
        
        if risk_factors:
            weights = [0.30, 0.25, 0.15, 0.10, 0.05, 0.05, 0.10]
            weights = weights[:len(risk_factors)]
            total_weight = sum(weights)
            weights = [w / total_weight for w in weights]
            
            risk_score = sum(factor * weight for factor, weight in zip(risk_factors, weights))
            mapped['risk_score'] = risk_score.clip(0, 100)
        else:
            mapped['risk_score'] = 50
        
        mapped['dataset_source'] = dataset_name
        
        return mapped
    
    feature_mappings = {
        'age': ['Age', 'age'],
        'high_chol': ['HighChol', 'high_chol', 'high_cholesterol', 'HighCholesterol'],
        'heart_disease': ['HeartDiseaseorAttack', 'heart_disease', 'HeartDisease'],
        'stroke': ['Stroke', 'stroke'],
        'phys_activity': ['PhysActivity', 'physical_activity', 'PhysActivity', 'exercise'],
        'gen_health': ['GenHlth', 'general_health', 'GenHealth'],
        'ment_health': ['MentHlth', 'mental_health', 'MentHealth'],
        'phys_health': ['PhysHlth', 'physical_health', 'PhysHealth'],
        'diff_walk': ['DiffWalk', 'difficulty_walking', 'DiffWalk'],
        'sex': ['Sex', 'sex', 'gender'],
        'diabetes': ['Diabetes_012', 'Diabetes_binary', 'diabetes', 'Diabetes'],
    }
    
    for target_col, possible_cols in feature_mappings.items():
        for col in possible_cols:
            if col in df.columns:
                mapped[target_col] = df[col]
                break
    
    if dataset_name == 'main' and 'risk_score' in df.columns:
        for col in df.columns:
            if col not in ['user_id', 'date', 'dataset_source']:
                if col not in mapped.columns:
                    mapped[col] = df[col]
    else:
        risk_factors = []
        if 'high_bp' in mapped.columns:
            risk_factors.append(mapped['high_bp'] * 80)
        if 'high_chol' in mapped.columns:
            risk_factors.append(mapped['high_chol'] * 70)
        if 'heart_disease' in mapped.columns:
            risk_factors.append(mapped['heart_disease'] * 90)
        if 'stroke' in mapped.columns:
            risk_factors.append(mapped['stroke'] * 90)
        if 'gen_health' in mapped.columns:
            risk_factors.append((6 - mapped['gen_health']) / 5.0 * 100)
        if 'diabetes' in mapped.columns:
            risk_factors.append(mapped['diabetes'] * 50)
        
        if risk_factors:
            mapped['risk_score'] = pd.concat(risk_factors, axis=1).mean(axis=1)
        else:
            mapped['risk_score'] = 50
    
    mapped['dataset_source'] = dataset_name
    
    return mapped

df = load_and_map_datasets(DATASET_PATHS)

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())


In [ ]:
# ==============================================================================
# DATA PREPROCESSING & FEATURE ENGINEERING FOR MULTI-RISK PREDICTION
# ==============================================================================

def preprocess_for_multi_risk(df, feature_columns=None):
    """Preprocess data for training separate models for each risk type."""
    
    data = df.copy()
    
    # Remove rows with missing risk_score
    data = data.dropna(subset=['risk_score'])
    
    # Determine feature columns
    if feature_columns is None:
        exclude_cols = ['risk_score', 'dataset_source', 'user_id', 'date', 'bmi', 'high_bp', 'BMI', 'high_bp', 'Blood_Pressure_Systolic', 'Blood_Pressure_Diastolic']
        feature_columns = [col for col in data.columns if col not in exclude_cols]
    
    print(f"\nUsing {len(feature_columns)} features:")
    print(feature_columns[:10] if len(feature_columns) > 10 else feature_columns)
    
    # Separate features
    X = data[feature_columns].copy()
    
    # Handle missing values
    for col in X.columns:
        if X[col].dtype in ['int64', 'float64']:
            X[col] = X[col].fillna(X[col].mean())
        else:
            X[col] = X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0)
    
    # Encode categorical features
    label_encoders = {}
    for col in X.select_dtypes(include=['object']).columns:
        if col in feature_columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
            label_encoders[col] = le
    
    def categorize_risk(score_series):
        """Convert risk scores to categories: low, moderate, high."""
        return score_series.apply(lambda s: 
            'low' if s < 33 else ('moderate' if s < 66 else 'high')
        )
    
    # Create target variables for each risk type based on risk_score and health indicators
    risk_targets = {}
    risk_score = pd.to_numeric(data['risk_score'], errors='coerce')
    
    # Map risk_score and health indicators to individual risk types
    # Neuropathy: related to diabetes duration, glucose levels (use HbA1c and blood glucose if available)
    neuropathy_score = risk_score.copy()
    if 'hba1c' in data.columns:
        hba1c = pd.to_numeric(data['hba1c'], errors='coerce').fillna(7.0)
        # Higher HbA1c increases neuropathy risk
        hba1c_factor = ((hba1c - 5.0) / 5.0 * 25).clip(0, 40)
        neuropathy_score = (neuropathy_score + hba1c_factor).clip(0, 100)
    if 'blood_glucose' in data.columns:
        glucose = pd.to_numeric(data['blood_glucose'], errors='coerce').fillna(120)
        # High glucose levels increase neuropathy risk
        glucose_factor = ((glucose - 100) / 50 * 15).clip(0, 30)
        neuropathy_score = (neuropathy_score + glucose_factor).clip(0, 100)
    risk_targets['neuropathy'] = categorize_risk(neuropathy_score)
    
    # Retinopathy: related to diabetes control (use HbA1c if available, otherwise gen_health proxy)
    retinopathy_score = risk_score.copy()
    if 'hba1c' in data.columns:
        hba1c = pd.to_numeric(data['hba1c'], errors='coerce').fillna(7.0)
        # Higher HbA1c significantly increases retinopathy risk
        hba1c_factor = ((hba1c - 5.0) / 5.0 * 30).clip(0, 50)
        retinopathy_score = (risk_score + hba1c_factor).clip(0, 100)
    elif 'gen_health' in data.columns:
        gen_health = pd.to_numeric(data['gen_health'], errors='coerce').fillna(3)
        # Poor health (5) increases retinopathy risk
        retinopathy_score = (risk_score + (6 - gen_health) * 10).clip(0, 100)
    risk_targets['retinopathy'] = categorize_risk(retinopathy_score)
    
    # Nephropathy: related to high BP, diabetes control (use HbA1c if available)
    nephropathy_score = risk_score.copy()
    if 'high_bp' in data.columns:
        high_bp = pd.to_numeric(data['high_bp'], errors='coerce').fillna(0)
        nephropathy_score = (risk_score + high_bp * 20).clip(0, 100)
    if 'hba1c' in data.columns:
        hba1c = pd.to_numeric(data['hba1c'], errors='coerce').fillna(7.0)
        # Higher HbA1c increases nephropathy risk
        hba1c_factor = ((hba1c - 5.0) / 5.0 * 25).clip(0, 40)
        nephropathy_score = (nephropathy_score + hba1c_factor).clip(0, 100)
    risk_targets['nephropathy'] = categorize_risk(nephropathy_score)
    
    # Cardiovascular: related to heart disease, stroke, high cholesterol
    cardio_score = risk_score.copy()
    if 'heart_disease' in data.columns:
        heart = pd.to_numeric(data['heart_disease'], errors='coerce').fillna(0)
        cardio_score = (cardio_score + heart * 25).clip(0, 100)
    if 'stroke' in data.columns:
        stroke = pd.to_numeric(data['stroke'], errors='coerce').fillna(0)
        cardio_score = (cardio_score + stroke * 20).clip(0, 100)
    if 'high_chol' in data.columns:
        high_chol = pd.to_numeric(data['high_chol'], errors='coerce').fillna(0)
        cardio_score = (cardio_score + high_chol * 10).clip(0, 100)
    risk_targets['cardiovascular'] = categorize_risk(cardio_score)
    
    # Foot Problems: related to difficulty walking, diabetes control, circulation (use HbA1c if available)
    foot_score = risk_score.copy()
    if 'diff_walk' in data.columns:
        diff_walk = pd.to_numeric(data['diff_walk'], errors='coerce').fillna(0)
        foot_score = (risk_score + diff_walk * 30).clip(0, 100)
    if 'phys_health' in data.columns:
        phys_health = pd.to_numeric(data['phys_health'], errors='coerce').fillna(0)
        # Higher physical health problems increase foot risk
        foot_score = (foot_score + phys_health * 2).clip(0, 100)
    if 'hba1c' in data.columns:
        hba1c = pd.to_numeric(data['hba1c'], errors='coerce').fillna(7.0)
        # Higher HbA1c increases foot problems risk (poor circulation)
        hba1c_factor = ((hba1c - 5.0) / 5.0 * 20).clip(0, 30)
        foot_score = (foot_score + hba1c_factor).clip(0, 100)
    risk_targets['foot_problems'] = categorize_risk(foot_score)
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    print(f"\n✅ Preprocessing complete:")
    print(f"  Features shape: {X_scaled.shape}")
    print(f"\n  Risk type distributions:")
    for risk_type, y_risk in risk_targets.items():
        print(f"    {risk_type}: {y_risk.value_counts().to_dict()}")
    
    return X_scaled, risk_targets, feature_columns, scaler, label_encoders

# Preprocess data for multi-risk prediction
X, risk_targets, feature_columns, scaler, label_encoders = preprocess_for_multi_risk(
    df, 
    FEATURE_COLUMNS
)


In [ ]:
# ==============================================================================
# SPLIT DATA FOR TRAINING AND TESTING - Multiple Risk Types
# ==============================================================================

# Split data for each risk type
splits = {}
for risk_type, y_risk in risk_targets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_risk,
        test_size=CONFIG['test_size'],
        random_state=CONFIG['random_state'],
        stratify=y_risk
    )
    splits[risk_type] = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }
    print(f"✅ {risk_type}: {len(y_train)} train, {len(y_test)} test samples")

print(f"\nTotal training samples: {X_train.shape[0]}")
print(f"Total test samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")


In [ ]:
# ==============================================================================
# MODEL TRAINING - Train separate models for each risk type
# ==============================================================================

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def train_models_for_risk_type(X_train, y_train, X_test, y_test, risk_type):
    """Train multiple models for a specific risk type and return best one."""
    
    results = {}
    
    print(f"\n📊 Training models for {risk_type}...")
    
    # Define models
    model_configs = {
        'random_forest': RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=CONFIG['random_state'],
            n_jobs=-1
        ),
        'gradient_boosting': GradientBoostingClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            random_state=CONFIG['random_state']
        ),
        'logistic_regression': LogisticRegression(
            max_iter=1000,
            random_state=CONFIG['random_state'],
            n_jobs=-1
        )
    }
    
    # Add XGBoost only if available
    if XGBOOST_AVAILABLE and 'xgboost' in MODELS_TO_TRAIN:
        model_configs['xgboost'] = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=CONFIG['random_state'],
            eval_metric='mlogloss'
        )
    
    # Train each model
    for name, model in model_configs.items():
        if name not in MODELS_TO_TRAIN:
            continue
        
        try:
            # XGBoost requires numerical labels, other models can handle string labels
            if name == 'xgboost':
                label_encoder_y = LabelEncoder()
                y_train_encoded = label_encoder_y.fit_transform(y_train)
                y_test_encoded = label_encoder_y.transform(y_test)
                model.fit(X_train, y_train_encoded)
                y_pred_encoded = model.predict(X_test)
                y_pred = label_encoder_y.inverse_transform(y_pred_encoded)
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
            
            # Evaluate
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
            recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
            f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
            
            results[name] = {
                'model': model,
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1_score': f1,
                'predictions': y_pred
            }
            
            print(f"  ✅ {name}: Accuracy={accuracy:.4f} | F1={f1:.4f}")
            
        except Exception as e:
            print(f"  ❌ {name} Error: {e}")
    
    # Find best model
    if not results:
        raise ValueError(f"No models were trained successfully for {risk_type}!")
    
    best_model_name = max(results.keys(), key=lambda k: results[k]['f1_score'])
    print(f"  🏆 Best model: {best_model_name} (F1={results[best_model_name]['f1_score']:.4f})")
    
    return results, best_model_name

# Train models for each risk type
all_results = {}
best_models = {}

for risk_type, split_data in splits.items():
    print("\n" + "=" * 60)
    print(f"TRAINING MODELS FOR {risk_type.upper()}")
    print("=" * 60)
    
    results, best_name = train_models_for_risk_type(
        split_data['X_train'],
        split_data['y_train'],
        split_data['X_test'],
        split_data['y_test'],
        risk_type
    )
    
    all_results[risk_type] = results
    best_models[risk_type] = best_name


In [ ]:
# ==============================================================================
# MODEL EVALUATION - Detailed performance analysis for each risk type
# ==============================================================================

print("\n" + "=" * 60)
print("DETAILED EVALUATION RESULTS")
print("=" * 60)

# Evaluate each risk type
for risk_type in risk_targets.keys():
    print(f"\n{'='*60}")
    print(f"{risk_type.upper()} RISK MODEL EVALUATION")
    print(f"{'='*60}")
    
    split_data = splits[risk_type]
    results = all_results[risk_type]
    best_name = best_models[risk_type]
    best_result = results[best_name]
    
    print(f"\n📊 Classification Report for {risk_type} ({best_name}):")
    print(classification_report(split_data['y_test'], best_result['predictions']))
    
    print(f"\n📈 Confusion Matrix for {risk_type}:")
    cm = confusion_matrix(split_data['y_test'], best_result['predictions'])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {risk_type} ({best_name})')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # Print all model comparisons for this risk type
    print(f"\n📊 Model Comparison for {risk_type}:")
    comparison_df = pd.DataFrame({
        name: {
            'Accuracy': result['accuracy'],
            'Precision': result['precision'],
            'Recall': result['recall'],
            'F1 Score': result['f1_score']
        }
        for name, result in results.items()
    }).T
    print(comparison_df.round(4))

# Overall summary
print("\n" + "=" * 60)
print("OVERALL SUMMARY")
print("=" * 60)
summary_df = pd.DataFrame({
    risk_type: {
        'Best Model': best_models[risk_type],
        'F1 Score': all_results[risk_type][best_models[risk_type]]['f1_score'],
        'Accuracy': all_results[risk_type][best_models[risk_type]]['accuracy']
    }
    for risk_type in risk_targets.keys()
}).T
print(summary_df.round(4))


In [ ]:
# ==============================================================================
# SAVE MODELS - Save for API deployment (one model per risk type)
# ==============================================================================

# Prepare model data for saving - store models for each risk type
models_dict = {}
metrics_dict = {}

for risk_type in risk_targets.keys():
    best_name = best_models[risk_type]
    best_result = all_results[risk_type][best_name]
    models_dict[risk_type] = best_result['model']
    
    metrics_dict[risk_type] = {
        k: float(v) if isinstance(v, (np.integer, np.floating)) else str(v)
        for k, v in best_result.items() if k != 'model' and k != 'predictions'
    }

# Prepare model data for saving
model_data = {
    'models': models_dict,  # Dictionary of models, one per risk type
    'scaler': scaler,
    'label_encoders': label_encoders,
    'feature_columns': feature_columns,
    'model_type': 'classification',
    'model_names': best_models,  # Dictionary of best model names per risk type
    'metrics': metrics_dict,  # Dictionary of metrics per risk type
    'risk_thresholds': RISK_THRESHOLDS,
    'training_info': {
        'n_train_samples': len(X_train),
        'n_test_samples': len(X_test),
        'n_features': len(feature_columns),
        'datasets_used': list(DATASET_PATHS.keys()),
        'risk_types': list(risk_targets.keys())
    },
    'timestamp': pd.Timestamp.now().isoformat()
}

# Save model (API uses this filename)
model_path = OUTPUT_DIR / 'risk_forecast_trained.pkl'
joblib.dump(model_data, model_path)
print(f"\n✅ Model saved to: {model_path}")
print(f"   This file is used by the API: api/model_inference.py")
print(f"   Saved {len(models_dict)} risk type models: {list(models_dict.keys())}")

# Save training summary
summary = {
    'best_models': best_models,
    'all_results': {
        risk_type: {
            name: {
                k: float(v) if isinstance(v, (np.integer, np.floating)) else str(v)
                for k, v in result.items() if k != 'model' and k != 'predictions'
            }
            for name, result in all_results[risk_type].items()
        }
        for risk_type in risk_targets.keys()
    },
    'training_config': CONFIG,
    'feature_columns': feature_columns
}

summary_path = OUTPUT_DIR / 'risk_forecast_training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"✅ Training summary saved to: {summary_path}")

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)
for risk_type in risk_targets.keys():
    best_name = best_models[risk_type]
    best_result = all_results[risk_type][best_name]
    print(f"{risk_type}: {best_name} - Accuracy: {best_result['accuracy']:.4f}, F1: {best_result['f1_score']:.4f}")
print("=" * 60)


In [ ]:

print("Testing model loading...")
loaded_data = joblib.load(model_path)

print(f"✅ Model loaded successfully!")
print(f"   Model type: {loaded_data['model_type']}")
print(f"   Risk types: {list(loaded_data['models'].keys())}")
print(f"   Model names per risk type: {loaded_data['model_names']}")
print(f"   Features: {len(loaded_data['feature_columns'])}")
print(f"\n   Metrics per risk type:")
for risk_type, metrics in loaded_data['metrics'].items():
    accuracy = metrics.get('accuracy', 'N/A')
    f1 = metrics.get('f1_score', 'N/A')
    if isinstance(accuracy, (int, float)):
        accuracy = f"{accuracy:.4f}"
    if isinstance(f1, (int, float)):
        f1 = f"{f1:.4f}"
    print(f"     {risk_type}: Accuracy={accuracy}, F1={f1}")

print(f"\n📊 Test predictions on sample:")
if 'splits' in locals() and splits:
    first_risk_type = list(splits.keys())[0]
    test_sample = splits[first_risk_type]['X_test'][0:1]
    print(f"   Input shape: {test_sample.shape}")
    
    for risk_type in loaded_data['models'].keys():
        model = loaded_data['models'][risk_type]
        try:
            prediction = model.predict(test_sample)
            print(f"   {risk_type}: {prediction[0]}")
        except Exception as e:
            print(f"   {risk_type}: Error - {e}")
else:
    print("   (Test data not available - model can still be used by API)")

print("\n✅ Model is ready for API deployment!")
